In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Kaggle "House Prices: Advanced Regression Techniques" — Section 2 end-to-end
# EDA → Cleaning → Feature Engineering → Model Comparison → Best Model → Feature Importance → Submission
# Charts: matplotlib only, one plot per figure, default colors.

import os, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from typing import List, Dict, Any
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Optional boosters (skip gracefully if not installed)
xgb = None; lgb = None; CatBoostRegressor = None
try:
    import xgboost as xgb
except Exception:
    pass
try:
    import lightgbm as lgb
except Exception:
    pass
try:
    from catboost import CatBoostRegressor
except Exception:
    pass

# ---------------------------
# 1) Load data
# ---------------------------
def pick_first(names):
    for n in names:
        if os.path.exists(n):
            return n
    raise FileNotFoundError(names)

train_fp = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
test_fp  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"
sub_fp   = "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv"
desc_fp  = "/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt"

train = pd.read_csv(train_fp)
test  = pd.read_csv(test_fp)
sample_submission = pd.read_csv(sub_fp)

y = train["SalePrice"].copy()
X = train.drop(columns=["SalePrice"]).copy()

# ---------------------------
# 2) Initial EDA
# ---------------------------
def show_stats(y):
    s = pd.Series({
        "count": y.count(),
        "mean": y.mean(),
        "std": y.std(),
        "min": y.min(),
        "25%": y.quantile(0.25),
        "50%": y.median(),
        "75%": y.quantile(0.75),
        "max": y.max(),
        "skew": y.skew(),
        "log1p_skew": np.log1p(y).skew()
    })
    print("SalePrice stats:\n", s)

show_stats(y)

plt.figure()
plt.hist(y.values, bins=40)
plt.title("SalePrice Distribution")
plt.xlabel("SalePrice"); plt.ylabel("Count")
plt.show()

plt.figure()
plt.hist(np.log1p(y.values), bins=40)
plt.title("log1p(SalePrice) Distribution")
plt.xlabel("log1p(SalePrice)"); plt.ylabel("Count")
plt.show()

def top_missing(df, name, k=25):
    m = df.isna().mean().sort_values(ascending=False)
    t = m[m>0].head(k)
    print(f"\nTop missing in {name}:\n", t)
    return t

miss_train = top_missing(train, "train")
plt.figure()
plt.barh(np.arange(len(miss_train))[::-1], miss_train.values[::-1])
plt.yticks(np.arange(len(miss_train))[::-1], miss_train.index[::-1])
plt.title("Top 25 Missing Columns (train)")
plt.xlabel("Fraction Missing")
plt.tight_layout(); plt.show()

# ---------------------------
# 3) Cleaning & Feature Engineering
# ---------------------------

# "None" categoricals (true absence)
none_cats = [
    "Alley","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2",
    "FireplaceQu","GarageType","GarageFinish","GarageQual","GarageCond",
    "PoolQC","Fence","MiscFeature"
]
for df in [X, test]:
    for c in none_cats:
        if c in df.columns:
            df[c] = df[c].fillna("None")

# Neighborhood median impute for LotFrontage (robust across train+test)
combined = pd.concat([X[["Neighborhood","LotFrontage"]], test[["Neighborhood","LotFrontage"]]])
lf_by_nb = combined.groupby("Neighborhood")["LotFrontage"].median()
def impute_lf(df):
    s = df["LotFrontage"].copy()
    idx = s.isna()
    s.loc[idx] = df.loc[idx, "Neighborhood"].map(lf_by_nb)
    s.fillna(s.median(), inplace=True)
    return s
if "LotFrontage" in X.columns:
    X["LotFrontage"] = impute_lf(X)
    test["LotFrontage"] = impute_lf(test)

# Mode fills for a few low-missing categoricals
mode_fill = ["MSZoning","Functional","Electrical","KitchenQual","Exterior1st","Exterior2nd","SaleType","Utilities"]
for df in [X, test]:
    for c in mode_fill:
        if c in df.columns:
            df[c] = df[c].fillna(df[c].mode()[0])

# Zeros when NA means none
zero_fill = [
    "GarageYrBlt","GarageCars","GarageArea",
    "BsmtFinSF1","BsmtFinSF2","BsmtUnfSF","TotalBsmtSF",
    "MasVnrArea","BsmtFullBath","BsmtHalfBath"
]
for df in [X, test]:
    for c in zero_fill:
        if c in df.columns:
            df[c] = df[c].fillna(0)

# Treat MSSubClass as categorical
for df in [X, test]:
    if "MSSubClass" in df.columns:
        df["MSSubClass"] = df["MSSubClass"].astype(str)

# Classic outlier removal: huge GrLivArea but low SalePrice
mask = ~((X["GrLivArea"] > 4000) & (y < 300000))
if mask.sum() < len(X):
    print(f"Removing {(~mask).sum()} outlier rows")
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

# Ordinal encodings
qual_map = {"Ex":5,"Gd":4,"TA":3,"Fa":2,"Po":1,"None":0}
exp_map  = {"Gd":3,"Av":2,"Mn":1,"No":0,"None":0}
fin_map  = {"GLQ":6,"ALQ":5,"BLQ":4,"Rec":3,"LwQ":2,"Unf":1,"None":0}
pave_map = {"Y":2,"P":1,"N":0}
yn_map   = {"Y":1,"N":0}

ordinal_specs = {
    "ExterQual": qual_map, "ExterCond": qual_map, "BsmtQual": qual_map, "BsmtCond": qual_map,
    "HeatingQC": qual_map, "KitchenQual": qual_map, "FireplaceQu": qual_map,
    "GarageQual": qual_map, "GarageCond": qual_map, "PoolQC": qual_map,
    "BsmtExposure": exp_map, "BsmtFinType1": fin_map, "BsmtFinType2": fin_map,
    "PavedDrive": pave_map, "CentralAir": yn_map
}
def apply_ordinal(df):
    df = df.copy()
    for c, m in ordinal_specs.items():
        if c in df.columns:
            df[c] = df[c].map(m).astype(float)
    return df

X = apply_ordinal(X); test = apply_ordinal(test)

# Feature engineering
def add_features(df):
    d = df.copy()
    d["TotalSF"] = d.get("TotalBsmtSF", 0) + d.get("1stFlrSF", 0) + d.get("2ndFlrSF", 0)
    d["HouseAge"]   = d["YrSold"] - d["YearBuilt"]
    d["RemodAge"]   = d["YrSold"] - d["YearRemodAdd"]
    d["HasRemodel"] = (d["YearRemodAdd"] > d["YearBuilt"]).astype(int)
    d["GarageAge"]  = np.where(d.get("GarageYrBlt",0)>0, d["YrSold"] - d["GarageYrBlt"], np.nan)
    d["GarageAge"]  = d["GarageAge"].clip(lower=0).fillna(0)
    d["TotalBath"]  = d.get("FullBath",0) + 0.5*d.get("HalfBath",0) + d.get("BsmtFullBath",0) + 0.5*d.get("BsmtHalfBath",0)
    d["TotalPorchSF"] = d.get("OpenPorchSF",0)+d.get("EnclosedPorch",0)+d.get("3SsnPorch",0)+d.get("ScreenPorch",0)+d.get("WoodDeckSF",0)
    d["Has2ndFlr"]  = (d.get("2ndFlrSF",0) > 0).astype(int)
    d["HasBsmt"]    = (d.get("TotalBsmtSF",0) > 0).astype(int)
    d["HasGarage"]  = (d.get("GarageArea",0) > 0).astype(int)
    d["IsNewer"]    = (d["YearBuilt"] >= 2000).astype(int)
    return d

X = add_features(X); test = add_features(test)

# ---------------------------
# 4) Preprocessing
# ---------------------------
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_tf, numeric_cols),
    ("cat", categorical_tf, categorical_cols)
])

# ---------------------------
# 5) Models & CV
# ---------------------------
def rmse_log(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return float(np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2)))

rmse_log_scorer = make_scorer(rmse_log, greater_is_better=False)

def make_pipe(model):
    return Pipeline([
        ("pre", preprocessor),
        ("reg", TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1))
    ])

models: Dict[str, Any] = {
    "LinearRegression": LinearRegression(),
    "NeuralNet": MLPRegressor(hidden_layer_sizes=(128,64), activation="relu", early_stopping=True, random_state=42, max_iter=2000),
    "DecisionTree": DecisionTreeRegressor(random_state=42, min_samples_leaf=5),
    "RandomForest": RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42, learning_rate=0.05, n_estimators=1200, max_depth=3, subsample=0.9)
}

if xgb is not None:
    models["XGBoost"] = xgb.XGBRegressor(
        n_estimators=1200, learning_rate=0.05, max_depth=4, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=1.0, objective="reg:squarederror",
        random_state=42, n_jobs=-1
    )
if lgb is not None:
    models["LightGBM"] = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.03, num_leaves=31, subsample=0.8,
        colsample_bytree=0.8, random_state=42, n_jobs=-1
    )
if CatBoostRegressor is not None:
    models["CatBoost"] = CatBoostRegressor(depth=6, learning_rate=0.05, iterations=1500, random_seed=42, silent=True, loss_function="RMSE")

cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
fitted = {}
for name, mdl in models.items():
    pipe = make_pipe(mdl)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring=rmse_log_scorer)
    results.append({"Model": name, "CV_RMSE_Log": -scores.mean(), "Std": scores.std(), "Splits": cv.get_n_splits()})
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("CV_RMSE_Log")
print("\nModel comparison (5-fold CV, RMSE on log scale):")
print(results_df.to_string(index=False))

plt.figure()
plt.barh(results_df["Model"], results_df["CV_RMSE_Log"])
plt.xlabel("RMSE (log) — lower is better")
plt.title("Model Comparison (5-fold CV)")
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

best_name = results_df.iloc[0]["Model"]
print(f"\nBest model: {best_name}")
best_pipe = fitted[best_name].fit(X, y)

# ---------------------------
# 6) Feature importance (Top 10)
# ---------------------------
def get_feature_names(pre: ColumnTransformer) -> List[str]:
    out = []
    num = pre.transformers_[0][2]
    out.extend(num)
    cat_pipeline = pre.transformers_[1][1]
    ohe = cat_pipeline.named_steps["onehot"]
    cat_feats = pre.transformers_[1][2]
    out.extend(ohe.get_feature_names_out(cat_feats))
    return list(out)

feature_names = get_feature_names(best_pipe.named_steps["pre"])
reg = best_pipe.named_steps["reg"].regressor_

importance = None; method = None
# Linear coefficients
if hasattr(reg, "coef_") and reg.coef_ is not None:
    coefs = np.ravel(reg.coef_)
    importance = np.abs(coefs); method = "absolute coefficients"
# Tree importances
elif hasattr(reg, "feature_importances_"):
    importance = np.array(reg.feature_importances_); method = "feature_importances_"
# XGB/LGBM shortcuts
elif (xgb is not None and isinstance(reg, xgb.XGBRegressor)) or (lgb is not None and "LGBMRegressor" in type(reg).__name__):
    importance = np.array(reg.feature_importances_); method = "booster importances"
# Fallback: permutation importance (slower)
if importance is None:
    from sklearn.inspection import permutation_importance
    rng = np.random.RandomState(42)
    idx = rng.choice(len(X), size=min(300, len(X)), replace=False)
    Xs, ys = X.iloc[idx], y.iloc[idx]
    perm = permutation_importance(best_pipe, Xs, ys, scoring=rmse_log_scorer, n_repeats=5, random_state=42)
    importance = -perm.importances_mean  # negative scorer
    feature_names = get_feature_names(best_pipe.named_steps["pre"])
    method = "permutation importance"

imp_df = pd.DataFrame({"feature": feature_names, "importance": importance}).sort_values("importance", ascending=False).head(10)
print("\nTop 10 Feature Importance:", method)
print(imp_df.to_string(index=False))

plt.figure()
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
plt.title(f"Top 10 Feature Importance — {best_name} ({method})")
plt.xlabel("Importance")
plt.tight_layout(); plt.show()

# ---------------------------
# 7) Predict test & create submission
# ---------------------------
test_pred = best_pipe.predict(test)
test_pred = np.maximum(test_pred, 0)  # guard
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": test_pred})
submission.to_csv("submission.csv", index=False)
print('\nWrote submission file to "submission.csv" (Id,SalePrice).')

# ---------------------------
# 8) Bonus sanity plot (one split)
# ---------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=1)
tr_idx, va_idx = next(kf.split(X, y))
Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]
viz_pipe = fitted[best_name].fit(Xtr, ytr)
p = viz_pipe.predict(Xva)

plt.figure()
plt.scatter(np.log1p(yva), np.log1p(p), s=12)
plt.xlabel("True log1p(SalePrice)")
plt.ylabel("Pred log1p(SalePrice)")
plt.title(f"Validation: True vs Pred (log) — {best_name}")
lims = [min(np.log1p(yva).min(), np.log1p(p).min()), max(np.log1p(yva).max(), np.log1p(p).max())]
plt.plot(lims, lims)
plt.tight_layout(); plt.show()
